In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'svg'
import seaborn as sns
sns.set_style("ticks")
from geneinfo.plot import GenomeIdeogram, ChromIdeogram
from geneinfo.genelist import GeneList as glist
from tqdm.notebook import tqdm, trange

from vscodenb import cpu_monitor, pqdm, prange, set_vscode_theme
set_vscode_theme()

1.0


In [2]:
populations = [
        'CHB', 'JPT', 'CHS', 'CDX', 'KHV', #'CHD', 
        'CEU', 'TSI', 'GBR', 'FIN', 'IBS', 
        'YRI', 'LWK', 'GWD', 'MSL', 'ESN', 
        'ASW', 'ACB', 'MXL', 'PUR', 'CLM', 'PEL', 
        'GIH', 'PJL', 'BEB', 'STU', 'ITU'
        ]
chromosomes = [f'chr{x}' for x in range(1, 23)] + ['chrX'] 

In [4]:
df_list = []
for pop in pqdm(populations, leave=False):
    for chrom in pqdm(chromosomes, leave=False):

        inputPath = Path(f'../steps/relate/{chrom}/{pop}/haplotypes_demog_sele.sele')
        assert inputPath.exists(), inputPath
        with open(inputPath, 'r') as infile:
            df = pd.read_csv(infile, sep=' ')

            # fill in rs_ids fof chrX that is missing from the relate output
            if chrom == 'chrX':
                _df = pd.read_csv(f'../steps/relate/{chrom}/{pop}/haplotypes.mut', sep=';')
                _df['rs-id'] = [f'X:{pos}:{snp}' for pos, snp in zip(_df['pos_of_snp'], _df['ancestral_allele/alternative_allele'].str.replace('/', ':'))]
                _df_merged = df.merge(_df[['pos_of_snp', 'rs-id']], left_on='pos', right_on='pos_of_snp', how='left')
                df['rs_id'] = _df_merged['rs-id']

            df = df.loc[df[df.columns[2:]].min(axis=1) < -5]
            df['pop'] = pop
            df['chrom'] = chrom
            df_list.append(df)

df = pd.concat(df_list).reset_index()
df['chrom'] = pd.Categorical(df.chrom, categories=chromosomes, ordered=True)
df.sort_values(['chrom', 'pos', 'pop'], inplace=True)
df['variant_id'] = df.rs_id.str.replace(':', '-')
df = df[['chrom', 'pos', 'rs_id', 'variant_id', 'pop', 'when_DAF_is_half', 'when_mutation_has_freq2']]
# df.to_hdf('all_pops_sele.h5', key='df', format='table')
df.to_parquet('all_pops_sele.parquet', index=False)


populate variant id for chromosome X too:

In [5]:
df = pd.read_parquet('all_pops_sele.parquet')
#df = pd.read_hdf('all_pops_sele.h5')
df

,chrom,pos,rs_id,variant_id,pop,when_DAF_is_half,when_mutation_has_freq2
0,chr1,817341,1:817341:A:G,1-817341-A-G,CHS,-2.102790,-6.24528
1,chr1,819123,1:819123:G:A,1-819123-G-A,CHS,-2.102790,-6.24528
2,chr1,819123,1:819123:G:A,1-819123-G-A,MXL,-2.630620,-4.96580
3,chr1,849670,1:849670:G:A,1-849670-G-A,CHS,-1.607340,-5.42580
4,chr1,855085,1:855085:G:A,1-855085-G-A,YRI,-2.101570,-5.25291
...,...,...,...,...,...,...,...
106512,chrX,155999338,X:155999338:G:A,X-155999338-G-A,LWK,-1.402060,-6.64413
106513,chrX,155999435,X:155999435:C:G,X-155999435-C-G,ACB,-0.995721,-5.91133
106514,chrX,155999524,X:155999524:T:C,X-155999524-T-C,LWK,-0.973655,-5.00594
106515,chrX,155999818,X:155999818:C:T,X-155999818-C-T,GWD,-0.301155,-6.70790


In [ ]:
df.loc[df.variant_id != '.']

,chrom,pos,rs_id,variant_id,pop,when_DAF_is_half,when_mutation_has_freq2
2662,chr1,817341,1:817341:A:G,1-817341-A-G,CHS,-2.102790,-6.24528
2663,chr1,819123,1:819123:G:A,1-819123-G-A,CHS,-2.102790,-6.24528
41638,chr1,819123,1:819123:G:A,1-819123-G-A,MXL,-2.630620,-4.96580
2664,chr1,849670,1:849670:G:A,1-849670-G-A,CHS,-1.607340,-5.42580
17721,chr1,855085,1:855085:G:A,1-855085-G-A,YRI,-2.101570,-5.25291
...,...,...,...,...,...,...,...
104878,chr22,50483265,22:50483265:C:T,22-50483265-C-T,STU,-0.725086,-5.81052
96301,chr22,50551042,22:50551042:T:C,22-50551042-T-C,PEL,-1.355220,-3.51533
25102,chr22,50568667,22:50568667:T:C,22-50568667-T-C,LWK,-0.375965,-4.20560
34558,chr22,50597456,22:50597456:C:T,22-50597456-C-T,ESN,-2.096240,-4.44217


In [ ]:
import geneinfo.information as gi
records = []
for row in df.itertuples():
    if row.when_mutation_has_freq2 < -6:
        for name, *_ in gi.gene_coords_region(row.chrom, row.pos, row.pos+1, assembly='hg38'):
            record = (row.chrom, row.pos, name, row.pop, row.when_DAF_is_half, row.when_mutation_has_freq2)
            records.append(record)
df_genes = pd.DataFrame.from_records(records, columns=['chrom', 'pos', 'gene', 'pop', 'when_DAF_is_half', 'when_mutation_has_freq2'])
df_genes['in_nr_pops'] = df_genes.groupby(['chrom', 'pos', 'gene']).pop.transform('count')
df_genes.to_hdf('all_pops_sele_genes.h5', key='df', mode='w')

In [ ]:
all_genes = df_genes.gene.unique()

In [ ]:
df_genes = pd.read_hdf('all_pops_sele_genes.h5')

colors = {
        'C0': ['CHB', 'JPT', 'CHS', 'CDX', 'KHV'], 
        'C1': ['CEU', 'TSI', 'GBR', 'FIN', 'IBS'], 
        'C2': ['YRI', 'LWK', 'GWD', 'MSL', 'ESN'], 
        'C3': ['ASW', 'ACB', 'MXL', 'PUR', 'CLM', 'PEL'], 
        'C4': ['GIH', 'PJL', 'BEB', 'STU', 'ITU'],
}
color_map = {pop: col for col, pops in colors.items() for pop in pops}

_df = df_genes.groupby(['chrom', 'pos', 'gene']).when_mutation_has_freq2.min()

labels = []
for tup in _df.itertuples():
    if tup.when_mutation_has_freq2 < -9:
        labels.append((tup.chrom, tup.pos, tup.gene, color_map[tup.pop], -tup.when_mutation_has_freq2))

AttributeError: 'Series' object has no attribute 'itertuples'

In [ ]:
g = GenomeIdeogram(assembly='hg38') 
g.draw_chromosomes()
g.add_labels(labels, base=g.ideogram_base, min_height=g.ideogram_height*1.5)